# Bonus 08 — AutoGen message-driven teams

AutoGen made multi-agent conversations widely recognizable: named agents exchange messages, a team manager selects who speaks, tools appear as events, and termination conditions return control to software.

You will build and inspect a two-agent release-readiness team, account for its tool calls and tokens, save its state, restore it into fresh objects, and resume after human feedback.

> Current architecture note: AutoGen 0.7 is maintained for existing users, but Microsoft now directs new projects toward Microsoft Agent Framework. This lab teaches AutoGen systems you may inherit and the concepts you will migrate—not a default greenfield recommendation.


## 1. Learn — the conversation is a stateful protocol

```mermaid
flowchart LR
    U["Task from software"] --> M["RoundRobinGroupChat manager"]
    M --> D["drafter agent"]
    D -->|"TextMessage"| M
    M --> R["reviewer agent"]
    R -->|"ToolCallRequestEvent"| X["policy tool in Python"]
    X -->|"ToolCallExecutionEvent"| R
    R -->|"review + READY_FOR_HUMAN"| T["termination condition"]
    T --> O["TaskResult + stop reason"]
    O --> S["save_state"]
    S --> N["fresh team + load_state"]
    N --> H["resume with human feedback"]
```

The framework coordinates messages. The model still does not execute Python, approve a release, or decide whether your application continues. Software registers the tool, the AutoGen runtime invokes it, and a termination condition stops the team.


### First, know which “AutoGen” an example means

| Surface | Typical clues | How to treat it |
|---|---|---|
| AutoGen 0.2 | `ConversableAgent`, `UserProxyAgent`, `initiate_chat` | older API; do not copy it into this lab |
| AutoGen AgentChat 0.7 | `AssistantAgent`, `RoundRobinGroupChat`, `run_stream` | current AutoGen API used here; maintenance mode |
| AutoGen Core | event-driven runtime beneath AgentChat | use when you need lower-level actors, topics, and distributed runtime control |
| Microsoft Agent Framework | Microsoft's successor direction | evaluate first for a new Microsoft-centered production system |

Official status and migration guidance:

- [AutoGen repository and maintenance notice](https://github.com/microsoft/autogen)
- [AgentChat documentation](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/)
- [AutoGen to Microsoft Agent Framework migration guide](https://learn.microsoft.com/en-us/agent-framework/migration-guide/from-autogen/)

This distinction is an employable skill. Version-blind tutorials can teach an API that no longer represents the recommended architecture.


### Three layers, three kinds of control

| Layer | Gives you | Enterprise question |
|---|---|---|
| AgentChat | agents, teams, messages, termination, save/load | Is the high-level conversation pattern enough? |
| Core | routed agents and event-driven runtime | Do we need custom delivery, distributed workers, or actor-level control? |
| Extensions | model clients, executors, and other integrations | Which dependencies and external systems enter the trust boundary? |

A framework speeds up common coordination. It does not remove the need for authorization, idempotency, budgets, audit logs, data governance, and human approval.


### Adoption judgment

Use a framework when its message model, state lifecycle, and team patterns remove code you would otherwise maintain. Stay closer to an SDK or your own loop when the workflow is small, every transition must be obvious, or framework state and upgrade risk cost more than the convenience.

For an existing AutoGen system, first inventory:

1. old 0.2 versus AgentChat 0.7 APIs;
2. custom agents, tools, termination rules, state, and human-input points;
3. behavior that depends on message order or model choice;
4. evaluation traces that can prove a migration preserved behavior.

Do not start a rewrite from a feature table. Start from one representative workflow and its acceptance tests.


### Use the lab's isolated environment

From `bonus/08_autogen_message_driven_teams/`, run:

```bash
uv sync --locked
```

Then choose `bonus/08_autogen_message_driven_teams/.venv/bin/python` as the VS Code kernel. This lab pins AutoGen 0.7.5 and OpenAI 3.3.1 locally; it does not modify the core course environment.


## 2. Do — build the team with explicit boundaries


In [1]:
import json
import os
import warnings
from importlib.metadata import distributions, version

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_core.models import ModelFamily
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
assert env_path, "Repository-root .env not found"
load_dotenv(env_path)

MODEL_DEFAULT = os.environ["MODEL_DEFAULT"]
PRICE_INPUT = float(os.environ["PRICE_INPUT_PER_MILLION"])
PRICE_OUTPUT = float(os.environ["PRICE_OUTPUT_PER_MILLION"])

print("AutoGen AgentChat:", version("autogen-agentchat"))
print("AutoGen Core:", version("autogen-core"))
print("AutoGen Extensions:", version("autogen-ext"))
print("OpenAI SDK:", version("openai"))
print("Installed distributions:", len(list(distributions())))
print("Model from .env:", MODEL_DEFAULT)


AutoGen AgentChat: 0.7.5
AutoGen Core: 0.7.5
AutoGen Extensions: 0.7.5
OpenAI SDK: 3.3.1
Installed distributions: 58
Model from .env: gpt-5.4-nano


### Configure the provider boundary

The pinned AutoGen release predates this course's model alias, so its internal model table does not know the alias's capabilities. We provide that metadata explicitly. This is a concrete maintenance-mode cost: integration catalogs age even when the provider API still works.

The client uses Chat Completions with `max_completion_tokens`, no temperature, and `reasoning_effort="none"`. Do **not** set `parallel_tool_calls` globally: AutoGen would also forward it for the drafter, which has no tools, and the provider rejects that combination.

The API may resolve an alias to a dated snapshot. We suppress only AutoGen's known alias-mismatch warning so it does not obscure the event trace.


In [2]:
warnings.filterwarnings(
    "ignore",
    message=r"Resolved model mismatch:.*",
    category=UserWarning,
)

model_client = OpenAIChatCompletionClient(
    model=MODEL_DEFAULT,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": ModelFamily.GPT_5,
        "structured_output": True,
        "multiple_system_messages": True,
    },
    max_completion_tokens=500,
    reasoning_effort="none",
)

print({
    "model": MODEL_DEFAULT,
    "max_completion_tokens": 500,
    "reasoning_effort": "none",
    "temperature_set": False,
})


{'model': 'gpt-5.4-nano', 'max_completion_tokens': 500, 'reasoning_effort': 'none', 'temperature_set': False}


### Give the model one narrow, read-only tool

The release facts are safe, synthetic data. The tool returns a trusted policy decision for one exact change ID and records an application-side audit row. It cannot deploy, approve, write a file, or call another service.

The audit list is deliberately outside AutoGen. A framework event stream is useful evidence, but your application still owns durable audit and authorization.


In [3]:
CHANGE_RECORDS = {
    "CHG-77": {
        "owner": "data-platform",
        "summary": "Add customer_tier and backfill 1.2 million customer rows.",
        "risk_signals": ["schema change", "large backfill"],
        "rollback": "Restore compatibility view v41 and stop the backfill.",
        "window": "01:00 UTC",
    }
}

tool_audit = []


def evaluate_release_policy(change_id: str) -> str:
    """Return the trusted release-policy evaluation for one exact change ID."""
    record = CHANGE_RECORDS.get(change_id)
    tool_audit.append({
        "tool": "evaluate_release_policy",
        "change_id": change_id,
        "found": record is not None,
    })

    if record is None:
        return json.dumps({
            "change_id": change_id,
            "found": False,
            "decision": "BLOCK",
            "requirements": [],
        })

    return json.dumps({
        "change_id": change_id,
        "found": True,
        "decision": "HUMAN_REVIEW_REQUIRED",
        "requirements": [
            "named approver",
            "rollback owner",
            "post-change validation",
        ],
    })


### Separate roles without pretending they are authorities

The drafter turns supplied facts into a brief. The reviewer has the only tool and must call it once. Neither role can perform the release.

The factory is small but justified: AutoGen state is loaded into **fresh objects with matching names and structure**. Rebuilding the same team lets us prove state portability instead of accidentally continuing with the original Python objects.


In [4]:
DRAFTER_INSTRUCTIONS = """
Write a concise release-readiness brief from the task facts.
Include the change ID, owner, change, window, risks, and rollback.
Never use the exact marker READY_FOR_HUMAN.
""".strip()

REVIEWER_INSTRUCTIONS = """
Review the draft for enterprise release readiness.
Always call evaluate_release_policy exactly once for the task change ID.
Then state the decision, requirements, and that execution still needs a human.
End with the exact marker READY_FOR_HUMAN.
""".strip()


def build_release_team(client):
    drafter = AssistantAgent(
        name="drafter",
        model_client=client,
        system_message=DRAFTER_INSTRUCTIONS,
    )
    reviewer = AssistantAgent(
        name="reviewer",
        model_client=client,
        tools=[evaluate_release_policy],
        reflect_on_tool_use=True,
        max_tool_iterations=1,
        system_message=REVIEWER_INSTRUCTIONS,
    )

    marker_stop = TextMentionTermination(
        "READY_FOR_HUMAN",
        sources=["reviewer"],
    )
    hard_stop = MaxMessageTermination(max_messages=5)
    return RoundRobinGroupChat(
        participants=[drafter, reviewer],
        termination_condition=marker_stop | hard_stop,
        max_turns=4,
    )


team = build_release_team(model_client)
print({
    "team": type(team).__name__,
    "participants": ["drafter", "reviewer"],
    "termination": "reviewer marker OR five messages",
    "max_turns": 4,
})


{'team': 'RoundRobinGroupChat', 'participants': ['drafter', 'reviewer'], 'termination': 'reviewer marker OR five messages', 'max_turns': 4}


### Termination is control flow, not a polite prompt

The reviewer marker is the expected stop. The message cap and turn cap are independent safety ceilings. The marker condition is source-scoped, so a user or drafter cannot stop the team merely by repeating the text.

AutoGen termination conditions are stateful during a run and reset after the run finishes. They can be composed with `|` and `&`. A production design may also stop on token usage, time, cancellation, or a human-controlled application event.


### Run as a stream

`run_stream` exposes the protocol instead of hiding it behind a final string. The last yielded object is a `TaskResult`, not another chat message.


In [5]:
task = (
    "Prepare CHG-77 for human review. Facts: "
    + json.dumps(CHANGE_RECORDS["CHG-77"])
)

stream_items = []
async for item in team.run_stream(task=task):
    stream_items.append(item)

run_result = stream_items[-1]
assert isinstance(run_result, TaskResult)
print("Stream objects:", len(stream_items))
print("Stop reason:", run_result.stop_reason)


Stream objects: 6
Stop reason: Text 'READY_FOR_HUMAN' mentioned


## 3. Observe — inspect events, usage, and state


### Read the protocol trace

A tool interaction is not one magical step. The reviewer emits a request event, the runtime executes registered Python, and the result returns in an execution event before the reviewer writes its final message.


In [6]:
def content_view(content):
    if not isinstance(content, list):
        return content
    return [
        {
            "name": getattr(part, "name", None),
            "arguments": getattr(part, "arguments", None),
            "result": getattr(part, "content", None),
            "is_error": getattr(part, "is_error", None),
        }
        for part in content
    ]


event_rows = []
for item in stream_items:
    row = {
        "type": type(item).__name__,
        "source": getattr(item, "source", None),
        "content": content_view(getattr(item, "content", None)),
        "stop_reason": getattr(item, "stop_reason", None),
    }
    usage = getattr(item, "models_usage", None)
    if usage is not None:
        row["usage"] = {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
        }
    event_rows.append(row)
    print(json.dumps(row, indent=2, default=str))


{
  "type": "TextMessage",
  "source": "user",
  "content": "Prepare CHG-77 for human review. Facts: {\"owner\": \"data-platform\", \"summary\": \"Add customer_tier and backfill 1.2 million customer rows.\", \"risk_signals\": [\"schema change\", \"large backfill\"], \"rollback\": \"Restore compatibility view v41 and stop the backfill.\", \"window\": \"01:00 UTC\"}",
  "stop_reason": null
}
{
  "type": "TextMessage",
  "source": "drafter",
  "content": "## Release-Readiness Brief \u2014 CHG-77\n\n- **Change ID:** CHG-77  \n- **Owner:** data-platform  \n- **Change Summary:** Add **`customer_tier`** and backfill **1.2 million** customer rows.  \n- **Planned Window:** **01:00 UTC**  \n- **Risk Signals:**\n  - **Schema change** (potential downstream compatibility issues)\n  - **Large backfill** (performance/locking impact, extended runtime, potential partial-load behavior)\n- **Rollback Plan:**\n  - Restore **compatibility view v41**\n  - **Stop the backfill** to prevent further data change

In [7]:
event_types = [row["type"] for row in event_rows]
reviewer_texts = [
    row["content"]
    for row in event_rows
    if row["type"] == "TextMessage" and row["source"] == "reviewer"
]

assert event_types.count("ToolCallRequestEvent") == 1
assert event_types.count("ToolCallExecutionEvent") == 1
assert tool_audit == [{
    "tool": "evaluate_release_policy",
    "change_id": "CHG-77",
    "found": True,
}]
assert reviewer_texts and "READY_FOR_HUMAN" in reviewer_texts[-1]
assert "READY_FOR_HUMAN" in run_result.stop_reason
assert "Maximum number" not in run_result.stop_reason

print("Verified: one tool call, one tool result, reviewer-controlled stop.")


Verified: one tool call, one tool result, reviewer-controlled stop.


### Reconcile usage and estimate cost

Model usage belongs to individual emitted messages and also accumulates on the model client. The estimate below treats all prompt tokens as uncached because the framework does not expose a cached-input split here. It excludes any infrastructure, storage, or observability cost.


In [8]:
first_run_usage = model_client.total_usage()
estimated_cost = (
    first_run_usage.prompt_tokens * PRICE_INPUT
    + first_run_usage.completion_tokens * PRICE_OUTPUT
) / 1_000_000

usage_ledger = {
    "prompt_tokens": first_run_usage.prompt_tokens,
    "completion_tokens": first_run_usage.completion_tokens,
    "estimated_cost_usd": round(estimated_cost, 6),
    "assumption": "all prompt tokens priced as uncached input",
}
print(json.dumps(usage_ledger, indent=2))

assert first_run_usage.prompt_tokens > 0
assert first_run_usage.completion_tokens > 0


{
  "prompt_tokens": 888,
  "completion_tokens": 360,
  "estimated_cost_usd": 0.000628,
  "assumption": "all prompt tokens priced as uncached input"
}


### Save only after the team has stopped

Team state includes nested participant and manager state. It may contain conversation text, model context, tool results, and workflow position, so it needs the same access, retention, encryption, and deletion controls as other sensitive application data.

We inspect shape and size—not the full payload. AutoGen state is a snapshot for compatible code and object names; it is not a promise that every future framework version can load it.


In [9]:
saved_state = await team.save_state()
serialized_state = json.dumps(saved_state)

state_summary = {
    "top_level_keys": list(saved_state),
    "agent_names": list(saved_state["agent_states"]),
    "serialized_bytes": len(serialized_state.encode("utf-8")),
    "contains_api_key": os.environ["OPENAI_API_KEY"] in serialized_state,
}
print(json.dumps(state_summary, indent=2))

assert set(state_summary["agent_names"]) == {
    "drafter",
    "reviewer",
    "RoundRobinGroupChatManager",
}
assert state_summary["contains_api_key"] is False


{
  "top_level_keys": [
    "type",
    "version",
    "agent_states"
  ],
  "agent_names": [
    "drafter",
    "reviewer",
    "RoundRobinGroupChatManager"
  ],
  "serialized_bytes": 8462,
  "contains_api_key": false
}


### Reset, resume, and replay are different

- `reset()` clears a live team's conversation state.
- `save_state()` and `load_state()` move a snapshot into structurally compatible objects.
- A resumed model call is **not deterministic replay**. The model can phrase a new answer differently, and external tools may have changed.

For side-effecting tools, resumption also creates duplicate-execution risk. Use idempotency keys and durable application records. This lab's tool is read-only.


### What carries into a migration

The exact class names will change, but these design artifacts should survive:

| AutoGen artifact | Migration invariant |
|---|---|
| named agents and team manager | role ownership and message routing |
| tool request/execution events | authorization, arguments, result, and audit |
| termination conditions | explicit success and safety ceilings |
| team state | resumable workflow position and governed conversation data |
| event and usage assertions | behavioral acceptance tests |

That is why the lab inspects protocol objects instead of merely printing a polished final response.


## 4. Challenge — restore into fresh objects and resume

A human moves the release window from 01:00 to 02:00 UTC. Restore `saved_state` into a fresh, structurally identical team and ask it to reissue CHG-77 for review.

Acceptance criteria:

1. create the team with `build_release_team(model_client)`;
2. load `saved_state` before running;
3. clear `tool_audit` so this run has its own application audit;
4. stream the follow-up into `follow_up_items`;
5. keep the final `TaskResult` in `follow_up_result`;
6. the drafter must preserve CHG-77 context and use 02:00 UTC;
7. the reviewer must call the policy tool exactly once and return control to a human.

Do not edit the saved JSON or reuse the original `team`.


In [10]:
restored_team = build_release_team(model_client)
await restored_team.load_state(saved_state)

tool_audit.clear()
follow_up_task = (
    "Human feedback: move the CHG-77 window to 02:00 UTC. "
    "Reissue it for human review."
)

follow_up_items = []
async for item in restored_team.run_stream(task=follow_up_task):
    follow_up_items.append(item)

follow_up_result = follow_up_items[-1]


In [11]:
assert restored_team is not None, "Build a fresh team"
assert restored_team is not team, "Do not reuse the original team object"
assert isinstance(follow_up_result, TaskResult), "Keep the final TaskResult"
assert tool_audit == [{
    "tool": "evaluate_release_policy",
    "change_id": "CHG-77",
    "found": True,
}], "The resumed run must make exactly one audited policy lookup"

follow_up_event_types = [type(item).__name__ for item in follow_up_items]
assert follow_up_event_types.count("ToolCallRequestEvent") == 1
assert follow_up_event_types.count("ToolCallExecutionEvent") == 1

follow_up_messages = [
    item for item in follow_up_items
    if type(item).__name__ == "TextMessage"
]
drafter_follow_up = next(
    item.content for item in follow_up_messages
    if item.source == "drafter"
)
reviewer_follow_up = next(
    item.content for item in follow_up_messages
    if item.source == "reviewer"
)

assert "CHG-77" in drafter_follow_up
assert "02:00" in drafter_follow_up
assert "READY_FOR_HUMAN" in reviewer_follow_up
assert "READY_FOR_HUMAN" in follow_up_result.stop_reason

print(drafter_follow_up)
print()
print(reviewer_follow_up)
print()
print("Challenge passed:", follow_up_result.stop_reason)


## Release-Readiness Brief (Reissued) — CHG-77

- **Change ID:** CHG-77  
- **Owner:** data-platform  
- **Change Summary:** Add **`customer_tier`** and backfill **1.2 million** customer rows.  
- **Planned Window:** **02:00 UTC**  
- **Risk Signals:**
  - **Schema change** (potential downstream compatibility issues)
  - **Large backfill** (performance/locking impact, extended runtime, potential partial-load behavior)
- **Rollback Plan:**
  - Restore **compatibility view v41**
  - **Stop the backfill** to prevent further data changes

**Note:** Human review still required for the named approver, rollback owner, and post-change validation/verification steps.

### CHG-77 — Reissued for Human Review (Window Updated)

**Decision:** **HUMAN_REVIEW_REQUIRED**

**Updated requirement received:** Change the planned execution window to **02:00 UTC** (from 01:00 UTC).

**Requirements (must be satisfied before execution):**
1. **Named approver**: specify the exact individual who will approve CHG-7

In [12]:
await model_client.close()
print("Model client closed.")


Model client closed.


## Takeaway

AutoGen's valuable lesson is not “make agents talk.” It is that a team is a message protocol with observable events, explicit turn selection, termination, state, and provider integrations.

You can use AutoGen 0.7 to understand or maintain existing systems. For a new production system, compare Microsoft Agent Framework and thinner SDK or custom-loop options against the same workflow and acceptance tests. Choose the smallest layer that gives you the control you actually need.
